# Huynen decomposition: Python–C parity on ALOS-1

Compare the Huynen target matrix from Python and C-PolSARpro on the San Francisco ALOS-1 scattering matrix. Select `output_basis="pauli"` for T3 or `"lexicographic"` for C3, then rerun the notebook. Both implementations retain the 1e-30 epsilon offsets.

In [ ]:
%load_ext autoreload
%autoreload 2

import shutil
import subprocess
from pathlib import Path

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.decompositions import huynen
from polsarpro.dev.io import read_psp_bin
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
from polsarpro.io import open_netcdf_beam

executable = Path("/home/c_psp/Soft/bin/data_process_sngl/huynen_decomposition.exe")
input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1")

boxcar_size = (7, 7)
output_basis = "pauli"
iodf, prefix = {"pauli": ("S2T3", "T"), "lexicographic": ("S2C3", "C")}[output_basis]
output_dir = Path("/data/psp/res") / f"huynen_alos1_{output_basis}_c"
output_dir.mkdir(parents=True, exist_ok=True)

## Load the common input

Python reads the NetCDF scattering matrix while C reads the equivalent S2 bands. C runs without a validity mask to match the zero-padded averaging in Python. Python uses the shared direct-convolution boxcar; C uses recursive averaging, so small floating-point differences are expected.

In [ ]:
S = open_netcdf_beam(input_file)
nrows, ncols = S.sizes["y"], S.sizes["x"]
S

## Run C-PolSARpro

Run this cell manually to measure the C runtime. Output reading is performed later, outside the execution timing.

The Pauli path converts S2 directly to T3. The lexicographic path constructs C3, converts it to T3 for the core, and converts the target back to C3. In the inspected C source, that input conversion omits the trailing row halo, potentially causing discrepancies near block bottoms. Python's conversion does not have this issue. Use the Pauli path as the primary numerical reference; inspect spatial errors when evaluating the lexicographic path.

In [ ]:
%%time
memory_error_file = output_dir / "memory_error.txt"
command = [
    str(executable),
    "-id", str(input_dir),
    "-od", str(output_dir),
    "-iodf", iodf,
    "-nwr", str(boxcar_size[0]),
    "-nwc", str(boxcar_size[1]),
    "-ofr", "0",
    "-ofc", "0",
    "-fnr", str(nrows),
    "-fnc", str(ncols),
    "-errf", str(memory_error_file),
]
subprocess.run(command)

## Run Python

Run this cell manually to measure the Python runtime, including the lazy computation. The core operates in T3 for either output basis.

In [ ]:
%%time
with ProgressBar():
    out_py = huynen(S, boxcar_size=boxcar_size, output_basis=output_basis).compute()

## Numerical comparison

Compare all nine real-valued bands, splitting complex matrix elements into real and imaginary parts for the shared error metrics.

In [ ]:
# C does not write an output configuration file.
shutil.copy(input_dir / "config.txt", output_dir / "config.txt")
py_bands = {}
for name, values in out_py.data_vars.items():
    band = f"{prefix}{name[1:]}"
    if np.iscomplexobj(values):
        py_bands[f"{band}_real"] = values.real
        py_bands[f"{band}_imag"] = values.imag
    else:
        py_bands[band] = values
py_compare = xr.Dataset(py_bands)
c_compare = {name: read_psp_bin(output_dir / f"{name}.bin") for name in py_compare}

In [ ]:
summarize_metrics(py_compare, c_compare, short_titles=False, verbose=False)

## Inspect spatial error patterns

Display the diagonal powers and their differences. For off-diagonal elements, select their real or imaginary bands from `py_compare` and `c_compare`. Pay attention to image boundaries and, for lexicographic output, block bottoms.

In [ ]:
diagonals = [f"{prefix}{index}{index}" for index in (1, 2, 3)]
visualize_errors(
    py_compare[diagonals],
    {name: c_compare[name] for name in diagonals},
    clip=True,
    sub_az=16,
)